# Phi Native Implementation Demo

This notebook demonstrates the native Python implementation of integrated information (phi) using pyspi functions directly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyspi.calculator import Calculator
from pyspi.lib.phi_native import phi_comp
import tempfile
import yaml
import time

print("Libraries imported successfully!")

## 1. Direct phi_native Function Usage

In [ ]:
# Generate test data
np.random.seed(42)
data = np.random.randn(2, 100)  # 2 variables, 100 time points

# Set up parameters for phi computation
Z = np.array([1, 2])  # Partition
params = {"tau": 1}

# Test different phi configurations
configs = [
    {"type_of_phi": "star", "normalization": 0},
    {"type_of_phi": "star", "normalization": 1},
    {"type_of_phi": "Geo", "normalization": 0},
    {"type_of_phi": "Geo", "normalization": 1}
]

print("Direct phi_native function results:")
print("=" * 50)

for i, config in enumerate(configs):
    options = {**config, "type_of_dist": "Gauss"}
    
    start_time = time.time()
    phi_value = phi_comp(data, Z, params, options)
    end_time = time.time()
    
    print(f"Config {i+1}: phi_{config['type_of_phi']} (norm={config['normalization']})")
    print(f"  Value: {phi_value:.6f}")
    print(f"  Time: {(end_time-start_time)*1000:.2f} ms")
    print()

## 2. Calculator with Configuration

In [ ]:
# Create phi configuration
config = {
    '.statistics.infotheory': {
        'IntegratedInformation': {
            'labels': ['undirected', 'nonlinear', 'unsigned', 'bivariate', 'time-dependent'],
            'configs': [
                {'phitype': 'star'},
                {'phitype': 'star', 'normalization': 1},
                {'phitype': 'Geo'},
                {'phitype': 'Geo', 'normalization': 1}
            ]
        }
    }
}

# Create temporary config file
with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    yaml.dump(config, f)
    config_path = f.name

# Use Calculator with configuration
np.random.seed(42)
test_data = np.random.randn(3, 200)  # 3 variables, 200 time points

print("Creating Calculator with phi configuration...")
calc = Calculator(test_data, config=config_path)

print(f"Calculator initialized with {len(calc.spis)} SPI configurations")

# Compute phi values
print("\nComputing phi values...")
start_time = time.time()
calc.compute()
end_time = time.time()

print(f"Computation completed in {end_time - start_time:.4f} seconds")
print(f"Number of result tables: {len(calc.table)}")

# Clean up
import os
os.unlink(config_path)

## 3. Results Visualization

In [ ]:
# Display results
print("Phi computation results:")
print("=" * 60)

for table_name, table_data in calc.table.items():
    print(f"\n{table_name}:")
    print("-" * 40)
    print(table_data)

    # Show statistics - handle both Series and DataFrame
    if hasattr(table_data, 'select_dtypes'):
        # DataFrame case
        numeric_data = table_data.select_dtypes(include=[np.number])
        if not numeric_data.empty:
            finite_values = numeric_data.values[np.isfinite(numeric_data.values)]
        else:
            finite_values = []
    else:
        # Series case
        if np.issubdtype(table_data.dtype, np.number):
            finite_values = table_data.values[np.isfinite(table_data.values)]
        else:
            finite_values = []

    if len(finite_values) > 0:
        print(f"  Range: [{finite_values.min():.6f}, {finite_values.max():.6f}]")
        print(f"  Mean: {finite_values.mean():.6f}")
        print(f"  Std: {finite_values.std():.6f}")